# Fine-Tune LaBraM on LEAD Data

Step-by-step fine-tuning of pretrained [LaBraM](https://huggingface.co/braindecode/labram-pretrained) on the LEAD ADFTD-PS dataset for 3-class classification (CN / FTD / AD).

Each cell prints a sanity check so you can verify state before moving on. Kernel: `eeg312` (Python 3.12, braindecode 1.4.0).

Sister notebook (frozen + linear probe): `labram_inference.ipynb`.

In [1]:
import json, pathlib, time
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from braindecode.models import Labram

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import classification_report, accuracy_score
from sklearn.utils.class_weight import compute_class_weight

torch.manual_seed(42)
np.random.seed(42)

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"Device      : {device}")
print(f"PyTorch     : {torch.__version__}")
import braindecode
print(f"braindecode : {braindecode.__version__}")

Device      : mps
PyTorch     : 2.11.0
braindecode : 1.4.0


## 1. Load the LEAD dataset

Same loader as in `labram_inference.ipynb`. `X.dat` is `(N, C, T)` float32, `y.dat` is `(N, 3)` with columns `[label, subject_id, sample_rate]`.

In [2]:
DATA_ROOT = pathlib.Path("../data/lead")
LEAD_DIR  = DATA_ROOT / "L400" / "ADFTD-PS"

with open(LEAD_DIR / "meta.json") as f:
    meta = json.load(f)

N, T, C = meta["N"], meta["T"], meta["C"]
print(f"Dataset : {LEAD_DIR.name}")
print(f"Windows : {N}  |  Channels : {C}  |  Timepoints : {T}")
print(f"Sample rates present : {meta['SAMPLE_RATE_LIST']}")

X_all = np.fromfile(LEAD_DIR / "X.dat", dtype=np.float32).reshape(N, C, T)
y_all = np.fromfile(LEAD_DIR / "y.dat", dtype=np.float32).reshape(N, 3)

labels   = y_all[:, 0].astype(np.int64)
subj_ids = y_all[:, 1].astype(int)
srates   = y_all[:, 2].astype(int)

print(f"\nLabel distribution     : {dict(zip(*np.unique(labels, return_counts=True)))}")
print(f"Sample-rate distribution: {dict(zip(*np.unique(srates, return_counts=True)))}")
print(f"Subjects total          : {len(np.unique(subj_ids))}")

Dataset : ADFTD-PS
Windows : 45258  |  Channels : 19  |  Timepoints : 400
Sample rates present : [200, 100, 50]

Label distribution     : {0: 18606, 1: 16722, 2: 9930}
Sample-rate distribution: {50: 6391, 100: 12911, 200: 25956}
Subjects total          : 88


## 2. Filter to 200 Hz windows

LaBraM's pretrained patch size is 200 samples. At 200 Hz, our T=400 windows give exactly 2 patches per window. Keeping only 200 Hz windows avoids resampling.

In [3]:
TARGET_SR = 200

mask200 = srates == TARGET_SR
X = X_all[mask200]
y = labels[mask200]
subj = subj_ids[mask200]

print(f"Kept {X.shape[0]} / {N} windows at {TARGET_SR} Hz")
print(f"X shape : {X.shape}  (windows, channels, timepoints)")
print(f"Label distribution : {dict(zip(*np.unique(y, return_counts=True)))}")
print(f"Subjects (200 Hz)  : {len(np.unique(subj))}")

Kept 25956 / 45258 windows at 200 Hz
X shape : (25956, 19, 400)  (windows, channels, timepoints)
Label distribution : {0: 10664, 1: 9592, 2: 5700}
Subjects (200 Hz)  : 88


## 3. Subject-level train / val / test split

Split must be by **subject**, not by window — otherwise windows from the same person leak across splits and accuracy is artificially inflated. Stratified on the subject-level label so class proportions are preserved.

Aim: 70 % train / 15 % val / 15 % test of subjects.

In [4]:
unique_subj = np.unique(subj)
subj_labels = np.array([y[subj == s][0] for s in unique_subj])

split1 = StratifiedShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
train_idx, holdout_idx = next(split1.split(unique_subj, subj_labels))

split2 = StratifiedShuffleSplit(n_splits=1, test_size=0.50, random_state=42)
val_local, test_local = next(split2.split(unique_subj[holdout_idx], subj_labels[holdout_idx]))

train_subj = unique_subj[train_idx]
val_subj   = unique_subj[holdout_idx][val_local]
test_subj  = unique_subj[holdout_idx][test_local]

train_mask = np.isin(subj, train_subj)
val_mask   = np.isin(subj, val_subj)
test_mask  = np.isin(subj, test_subj)

print(f"Train: {train_mask.sum():>6d} windows  |  {len(train_subj):>3d} subjects")
print(f"Val  : {val_mask.sum():>6d} windows  |  {len(val_subj):>3d} subjects")
print(f"Test : {test_mask.sum():>6d} windows  |  {len(test_subj):>3d} subjects")
print(f"\nTrain label dist: {dict(zip(*np.unique(y[train_mask], return_counts=True)))}")
print(f"Val   label dist: {dict(zip(*np.unique(y[val_mask],   return_counts=True)))}")
print(f"Test  label dist: {dict(zip(*np.unique(y[test_mask],  return_counts=True)))}")

assert set(train_subj).isdisjoint(val_subj)
assert set(train_subj).isdisjoint(test_subj)
assert set(val_subj).isdisjoint(test_subj)
print("\nNo subject overlap across splits ✓")

Train:  18512 windows  |   61 subjects
Val  :   3703 windows  |   13 subjects
Test :   3741 windows  |   14 subjects

Train label dist: {0: 7324, 1: 7060, 2: 4128}
Val   label dist: {0: 1599, 1: 894, 2: 1210}
Test  label dist: {0: 1741, 1: 1638, 2: 362}

No subject overlap across splits ✓


## 4. PyTorch Datasets & DataLoaders

In [5]:
class LeadDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).long()

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        return self.X[i], self.y[i]

BATCH_SIZE = 32

train_ds = LeadDataset(X[train_mask], y[train_mask])
val_ds   = LeadDataset(X[val_mask],   y[val_mask])
test_ds  = LeadDataset(X[test_mask],  y[test_mask])

pin = (device == "cuda")
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=pin)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=pin)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=pin)

xb, yb = next(iter(train_loader))
print(f"Batch X : {tuple(xb.shape)}  dtype={xb.dtype}")
print(f"Batch y : {tuple(yb.shape)}  dtype={yb.dtype}  unique={yb.unique().tolist()}")
print(f"Steps/epoch: train={len(train_loader)}  val={len(val_loader)}  test={len(test_loader)}")

Batch X : (32, 19, 400)  dtype=torch.float32
Batch y : (32,)  dtype=torch.int64  unique=[0, 1, 2]
Steps/epoch: train=579  val=116  test=117


## 5. Load pretrained LaBraM and adapt to our window size

LaBraM was pretrained on 15 patches of 200 timepoints (= 3000 samples @ 200 Hz). Our windows are 400 timepoints = 2 patches, so we interpolate the temporal positional embedding from 16 (15 + CLS) down to 3 (2 + CLS).

The 19-channel 10-20 names below get looked up in the model's per-channel embedding table at forward time.

In [6]:
CH_NAMES = [
    "Fp1", "Fp2", "F7", "F3", "Fz", "F4", "F8",
    "T7",  "C3",  "Cz", "C4", "T8",
    "P7",  "P3",  "Pz", "P4", "P8",
    "O1",  "O2",
]

PRETRAINED_N_TIMES = 3000
ACTUAL_N_TIMES     = X.shape[-1]
PATCH_SIZE         = 200
n_classes          = int(len(np.unique(y)))

model = Labram.from_pretrained(
    "braindecode/labram-pretrained",
    n_chans=128,
    n_times=PRETRAINED_N_TIMES,
    n_outputs=n_classes,
    sfreq=TARGET_SR,
    chs_info=None,
)

# interpolate temporal positional embedding 16 -> 3 (2 patches + CLS)
old_te = model.temporal_embedding.data
new_n  = ACTUAL_N_TIMES // PATCH_SIZE + 1
new_te = torch.nn.functional.interpolate(
    old_te.permute(0, 2, 1), size=new_n, mode="linear", align_corners=False
).permute(0, 2, 1).contiguous()
model.temporal_embedding = nn.Parameter(new_te)

# patch metadata for our actual input size
model._n_times                 = ACTUAL_N_TIMES
model.patch_embed[0].n_times   = ACTUAL_N_TIMES
model.patch_embed[0].n_patchs  = ACTUAL_N_TIMES // PATCH_SIZE

model = model.to(device)

n_params = sum(p.numel() for p in model.parameters())
n_train  = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Temporal embedding: {list(old_te.shape)} → {list(new_te.shape)}")
print(f"Total params       : {n_params:,}")
print(f"Trainable params   : {n_train:,}  (all unfrozen for full fine-tuning)")
print(f"Classes            : {n_classes}  (0=CN, 1=FTD, 2=AD)")

Temporal embedding: [1, 16, 200] → [1, 3, 200]
Total params       : 5,817,939
Trainable params   : 5,817,939  (all unfrozen for full fine-tuning)
Classes            : 3  (0=CN, 1=FTD, 2=AD)


## 6. Forward-pass sanity check

Make sure the model runs end-to-end on a real batch before we kick off training.

In [7]:
model.eval()
with torch.no_grad():
    xb_dev = xb.to(device)
    logits = model(xb_dev, ch_names=CH_NAMES)
print(f"Logits shape : {tuple(logits.shape)}  (batch, classes)")
print(f"Logits row 0 : {logits[0].detach().cpu().numpy()}")
print(f"argmax row 0 : {int(logits[0].argmax())}  (random before training)")

Logits shape : (32, 3)  (batch, classes)
Logits row 0 : [-4.7293234  -0.24007307  1.1499734 ]
argmax row 0 : 2  (random before training)


## 7. Loss, optimizer, scheduler

- **Class-weighted cross-entropy**: AD has the fewest windows, so balanced weights.
- **AdamW** with cosine LR schedule.
- Single LR for everything to start. To use a smaller backbone LR / larger head LR later, group params by name and pass two `lr` groups to AdamW.

In [8]:
class_weights = compute_class_weight("balanced", classes=np.arange(n_classes), y=y[train_mask])
class_weights = torch.tensor(class_weights, dtype=torch.float32, device=device)
print(f"Class weights: {class_weights.cpu().numpy()}")

criterion = nn.CrossEntropyLoss(weight=class_weights)

print("\nTop-level modules in LaBraM:")
for name, _ in model.named_children():
    print(f"  {name}")

LR           = 1e-4
WEIGHT_DECAY = 0.05
EPOCHS       = 5
GRAD_CLIP    = 1.0

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)

total_steps = EPOCHS * len(train_loader)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps)

print(f"\nLR={LR}  WD={WEIGHT_DECAY}  Epochs={EPOCHS}  Steps/epoch={len(train_loader)}  Total steps={total_steps}")

Class weights: [0.84252685 0.8740321  1.494832  ]

Top-level modules in LaBraM:
  patch_embed
  pos_drop
  blocks
  norm
  final_layer

LR=0.0001  WD=0.05  Epochs=5  Steps/epoch=579  Total steps=2895


## 8. Train + validate

Saves the best state-dict by validation accuracy in memory (not to disk yet).

In [ ]:
def run_epoch(loader, train):
    model.train(train)
    total_loss, total_correct, total_n = 0.0, 0, 0
    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        with torch.set_grad_enabled(train):
            logits = model(xb, ch_names=CH_NAMES)
            loss   = criterion(logits, yb)

            if train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                optimizer.step()
                scheduler.step()

        total_loss    += loss.item() * yb.size(0)
        total_correct += (logits.argmax(-1) == yb).sum().item()
        total_n       += yb.size(0)

    return total_loss / total_n, total_correct / total_n

best_val_acc = 0.0
best_state   = None
history      = []

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    train_loss, train_acc = run_epoch(train_loader, train=True)
    val_loss,   val_acc   = run_epoch(val_loader,   train=False)
    history.append((train_loss, train_acc, val_loss, val_acc))

    star = ""
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state   = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        star = "  *"

    print(
        f"Epoch {epoch}/{EPOCHS}  "
        f"train loss={train_loss:.4f} acc={train_acc:.3f}  |  "
        f"val loss={val_loss:.4f} acc={val_acc:.3f}  "
        f"({time.time() - t0:.1f}s){star}"
    )

print(f"\nBest val acc: {best_val_acc:.3f}")

Epoch 1/5  train loss=1.1582 acc=0.324  |  val loss=1.1231 acc=0.276  (114.9s)  *


## 9. Evaluate on the test set with the best weights

In [ ]:
if best_state is not None:
    model.load_state_dict(best_state)
model.eval()

all_logits, all_y = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device, non_blocking=True)
        logits = model(xb, ch_names=CH_NAMES)
        all_logits.append(logits.cpu().numpy())
        all_y.append(yb.numpy())

logits_test = np.concatenate(all_logits)
y_test_arr  = np.concatenate(all_y)
y_pred      = logits_test.argmax(-1)

class_names = ["CN", "FTD", "AD"][:n_classes]

print(f"Window-level accuracy : {accuracy_score(y_test_arr, y_pred):.3f}\n")
print(classification_report(y_test_arr, y_pred, target_names=class_names))

## 10. Subject-level majority vote

Aggregates window predictions to one prediction per held-out subject.

In [ ]:
test_subj_arr = subj[test_mask]
subj_true, subj_pred = [], []

for s in sorted(np.unique(test_subj_arr)):
    s_mask = test_subj_arr == s
    subj_true.append(int(y_test_arr[s_mask][0]))
    subj_pred.append(int(Counter(y_pred[s_mask]).most_common(1)[0][0]))

subj_acc = accuracy_score(subj_true, subj_pred)
print(f"Subject-level accuracy (majority vote) : {subj_acc:.3f}  ({len(subj_true)} subjects)\n")
print(classification_report(subj_true, subj_pred, target_names=class_names))

## 11. Save the fine-tuned model

In [ ]:
out_dir = pathlib.Path("../models")
out_dir.mkdir(exist_ok=True)
out_path = out_dir / "labram_finetuned_adftdps.pt"

torch.save({
    "state_dict"   : model.state_dict(),
    "ch_names"     : CH_NAMES,
    "n_classes"    : n_classes,
    "n_times"      : ACTUAL_N_TIMES,
    "patch_size"   : PATCH_SIZE,
    "sfreq"        : TARGET_SR,
    "best_val_acc" : best_val_acc,
}, out_path)
print(f"Saved to {out_path.resolve()}")